# 3D Cube Animation Using Rotation Matrices

This notebook demonstrates how to construct and animate a **3D cube** using Python, NumPy, and Matplotlib.  
The animation is based on **linear algebra transformations**, specifically **rotation matrices**.

---

# 1. Mathematical Representation of a Cube

A cube has **8 vertices**.

We define a cube centered at the origin with coordinates:

Bottom square (z = -1)

(-1, -1, -1)  
( 1, -1, -1)  
( 1,  1, -1)  
(-1,  1, -1)

Top square (z = 1)

(-1, -1,  1)  
( 1, -1,  1)  
( 1,  1,  1)  
(-1,  1,  1)

---

# 2. Vertex Matrix Representation

Instead of storing vertices separately, we store them in a **matrix form**.

Each column represents **one vertex**.

\[
P =
\begin{bmatrix}
-1 & 1 & 1 & -1 & -1 & 1 & 1 & -1 \\
-1 & -1 & 1 & 1 & -1 & -1 & 1 & 1 \\
-1 & -1 & -1 & -1 & 1 & 1 & 1 & 1
\end{bmatrix}
\]

Interpretation:

| Index | Vertex |
|------|------|
| 0 | (-1,-1,-1) |
| 1 | (1,-1,-1) |
| 2 | (1,1,-1) |
| 3 | (-1,1,-1) |
| 4 | (-1,-1,1) |
| 5 | (1,-1,1) |
| 6 | (1,1,1) |
| 7 | (-1,1,1) |

---

# 3. Cube Edges

Edges define **which vertices are connected by lines**.

A cube has **12 edges**.

Bottom face:

(0,1), (1,2), (2,3), (3,0)

Top face:

(4,5), (5,6), (6,7), (7,4)

Vertical edges:

(0,4), (1,5), (2,6), (3,7)

These pairs represent **connections between vertices**.

---

# 4. Why Use Vertex Indices Instead of Coordinates?

Edges are stored as **pairs of vertex indices**, not coordinates.

Example:

(0,1)

means:

connect vertex 0 → vertex 1.

This approach is powerful because:

- vertices can move (during rotation)
- edges automatically update
- geometry structure remains constant

This is the same idea used in **3D mesh models**.

---

# 5. Rotation of the Cube

Rotation in 3D space is achieved using **rotation matrices**.

Rotation around the X-axis:

\[
$R_x(\theta)$=
\begin{bmatrix}
1 & 0 & 0 \\
0 & \cos\theta & -\sin\theta \\
0 & \sin\theta & \cos\theta
\end{bmatrix}
\]

Rotation around the Z-axis:

\[
$R_z (\theta)$=
\begin{bmatrix}
\cos\theta & -\sin\theta & 0 \\
\sin\theta & \cos\theta & 0 \\
0 & 0 & 1
\end{bmatrix}
\]

---

# 6. Rotating the Cube

The vertex matrix is rotated using matrix multiplication:

\[
P' = R P
\]

where:

- \(P\) = original vertex matrix
- \(R\) = rotation matrix
- \(P'\) = rotated vertices

Because the matrix multiplication is applied to the **entire matrix**, all vertices rotate **simultaneously**.

---

# Summary

A 3D object can be represented as:

Vertices → positions of points  
Edges → connections between points  

Animation is created by applying **time-dependent transformations** to the vertex matrix.

\[
P'(t) = R(t)P
\]


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
plt.rcParams["animation.embed_limit"] = 50  # MB

fig, ax = plt.subplots(
    figsize=(6, 6),
    dpi=100,
    subplot_kw={"projection": "3d"}
)

ax.set_xlim(-2, 2)
ax.set_ylim(-2, 2)
ax.set_zlim(-2, 2)

ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_zlabel("Z")

ax.view_init(elev=25, azim=30)

fig.patch.set_facecolor("lightgray") #whole canvas
ax.set_facecolor("cyan") #background of 3D boxes
ax.xaxis.pane.set_facecolor("lightblue") #background of x-axis plane
ax.yaxis.pane.set_facecolor("lightgreen") #background of y-axis plane
ax.zaxis.pane.set_facecolor("lightyellow") #background of z-axis plane
ax.grid(False) #turn off grid

# cube vertices
P = np.array([
[-1, 1, 1,-1,-1, 1, 1,-1],
[-1,-1, 1, 1,-1,-1, 1, 1],
[-1,-1,-1,-1, 1, 1, 1, 1]
])


# cube edges
edges = [
(0,1),(1,2),(2,3),(3,0),
(4,5),(5,6),(6,7),(7,4),
(0,4),(1,5),(2,6),(3,7)
]

lines = []
for e in edges:
    line, = ax.plot(
        [P[0,e[0]], P[0,e[1]]],
        [P[1,e[0]], P[1,e[1]]],
        [P[2,e[0]], P[2,e[1]]],
        color='blue'
    )
    lines.append(line)

def Rx(t):
    return np.array([
        [1,0,0],
        [0,np.cos(t),-np.sin(t)],
        [0,np.sin(t), np.cos(t)]
    ])

def Rz(t):
    return np.array([
        [np.cos(t),-np.sin(t),0],
        [np.sin(t), np.cos(t),0],
        [0,0,1]
    ])

def update(frame):

    theta = frame * 0.04

    R = Rz(theta) @ Rx(theta)
    P_rot = R @ P

    for line,edge in zip(lines,edges):
        i,j = edge
        line.set_data(
            [P_rot[0,i],P_rot[0,j]],
            [P_rot[1,i],P_rot[1,j]]
        )

        line.set_3d_properties(
            [P_rot[2,i],P_rot[2,j]]
        )

        ax.set_title(f"rigid body rotation (cube)\nangle={theta:.2f} rad")

    return lines

ani = FuncAnimation(fig, update, frames=200, interval=60, blit=True)
plt.close(fig)
HTML(ani.to_jshtml())

#ave the animation as an MP4 file
# ani.save(
#     "/home/shubh/projects/3d-anim/assets/videos/cube_rotation.mp4",
#     writer="ffmpeg",
#     fps=24,
#     dpi=200,
#     bitrate=2400
# )